In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import ast

## 1. Load the Master Dataset

In [2]:
df = pd.read_csv('../../data/test-final/master_dataset_with_attention_embeddings.csv')
print("Master dataset loaded. Here are the first 5 rows:")
display(df.head())

Master dataset loaded. Here are the first 5 rows:


,transcript_id,jd_id,candidate_id,segments,overall_scores.technical_skills,overall_scores.experience,overall_scores.problem_solving,overall_scores.communication,overall_scores.cultural_fit,overall_scores.overall_suitability,...,Overall_Suitability,Recommendation,Interviewer_Comments,Ratings.Technical_Proficiency,Ratings.Problem_Solving_Ability,Ratings.Communication_Skills,Ratings.Cultural_Team_Fit,Ratings.Adaptability_Learning,segment_count,transcript_embedding
0,t001,ds_jd_01,c001,"[{""segment_id"": ""t001_a01"", ""speaker"": ""Candid...",0.95,0.92,0.90,0.94,0.93,0.93,...,5.0,Hire,maria demonstrates exceptional technical skill...,5.0,5.0,5.0,5.0,5.0,NaN,"[-0.01575796357294339, -0.01892936692332188, 0..."
1,t002,ds_jd_01,c002,"[{""segment_id"": ""t002_a01"", ""speaker"": ""Candid...",0.92,0.88,0.90,0.85,0.89,0.89,...,5.0,Hire,david showcases strong technical expertise mac...,5.0,5.0,5.0,4.0,5.0,NaN,"[-0.003749942939705951, -0.01209289917532156, ..."
2,t003,ds_jd_01,c003,"[{""segment_id"": ""t003_a01"", ""speaker"": ""Candid...",0.75,0.70,0.78,0.80,0.72,0.75,...,4.0,Consider,aisha solid technical skills experience buildi...,4.0,4.0,4.0,4.0,4.0,NaN,"[-0.025586010115566844, -0.001228203945624485,..."
3,t004,ds_jd_01,c004,"[{""segment_id"": ""t004_a01"", ""speaker"": ""Candid...",0.95,0.95,0.90,0.95,0.90,0.93,...,5.0,Hire,priya highly skilled machine learning data pip...,5.0,5.0,5.0,5.0,5.0,NaN,"[-0.02135167594809402, 0.0038519784337327687, ..."
4,t005,ds_jd_01,c005,"[{""segment_id"": ""t005_a01"", ""speaker"": ""Candid...",0.92,0.91,0.89,0.88,0.90,0.90,...,5.0,Hire,raj possesses extensive technical expertise ma...,5.0,5.0,5.0,5.0,5.0,NaN,"[-0.01619285595759264, -0.004747431692865577, ..."


## 2. Build Skill Vocabulary from `extracted_skills`

In [3]:
def build_skill_vocab(skill_series):
    skills = set()
    for skill_str in skill_series.dropna():
        # Split by comma and add stripped skill names to the set
        for s in skill_str.split(','):
            skills.add(s.strip())
    return sorted(list(skills))

skill_vocab = build_skill_vocab(df['extracted_skills'])
print("Skill Vocabulary (Unique Skills):")
print(skill_vocab)

Skill Vocabulary (Unique Skills):
['.NET', 'API design', 'Agile', 'Artificial Intelligence', 'Azure', 'CI CD DevOps', 'Cloud Deployment', 'Communication', 'Data Analysis', 'Data Engineering', 'Data Pipelines', 'Data Preprocessing', 'Data Visualization', 'Deep Learning', 'Java', 'JavaScript', 'Machine Learning', 'Microsoft technologies', 'NodeJS', 'Predictive Modeling', 'Problem Solving', 'Python', 'React', 'SQL', 'Statistical Analysis', 'WinForms', 'cloud platforms', 'code patching', 'documentation', 'full stack development', 'mentorship', 'problem solving', 'system design', 'troubleshooting']


## 3. Encode JD Extracted Skills as Binary Indicators

In [5]:
def encode_skills(skill_str, vocab):
    if pd.isnull(skill_str):
        return [0] * len(vocab)
    skills = [s.strip() for s in skill_str.split(',')]
    return [1 if skill in skills else 0 for skill in vocab]

df['skills_vector'] = df['extracted_skills'].apply(lambda x: encode_skills(x, skill_vocab))
print("Sample encoded skills vector for first row:")
print(df['skills_vector'].iloc[0])


Sample encoded skills vector for first row:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## 4. Normalize Questionnaire Ratings
We normalize the numeric ratings using MinMaxScaler.

In [6]:
rating_cols = [
    'Ratings.Technical_Proficiency',
    'Ratings.Problem_Solving_Ability',
    'Ratings.Communication_Skills',
    'Ratings.Cultural_Team_Fit',
    'Ratings.Adaptability_Learning'
]


In [7]:
# Print unique values before normalization
for col in rating_cols:
    print(f"Unique values in {col} before scaling: {df[col].unique()}")

scaler = MinMaxScaler()
df[rating_cols] = scaler.fit_transform(df[rating_cols])

print("\nNormalized ratings sample:")
display(df[rating_cols].head())

Unique values in Ratings.Technical_Proficiency before scaling: [5.  4.  2.  3.  1.  3.5 2.5 1.5]
Unique values in Ratings.Problem_Solving_Ability before scaling: [5.  4.  2.  1.  3.  3.5]
Unique values in Ratings.Communication_Skills before scaling: [5.  4.  3.  2.  3.5 2.5 1. ]
Unique values in Ratings.Cultural_Team_Fit before scaling: [5.  4.  3.  2.  1.  3.5 2.5]
Unique values in Ratings.Adaptability_Learning before scaling: [5.  4.  3.  2.  4.5 3.5 1.  2.5]

Normalized ratings sample:


,Ratings.Technical_Proficiency,Ratings.Problem_Solving_Ability,Ratings.Communication_Skills,Ratings.Cultural_Team_Fit,Ratings.Adaptability_Learning
0,1.00,1.00,1.00,1.00,1.00
1,1.00,1.00,1.00,0.75,1.00
2,0.75,0.75,0.75,0.75,0.75
3,1.00,1.00,1.00,1.00,1.00
4,1.00,1.00,1.00,1.00,1.00


## 5. One-Hot Encode the Categorical 'Recommendation' Column

In [8]:
df = pd.get_dummies(df, columns=['Recommendation'], prefix='rec')
print("Columns after one-hot encoding Recommendation:")
print(df.columns)


Columns after one-hot encoding Recommendation:
Index(['transcript_id', 'jd_id', 'candidate_id', 'segments',
       'overall_scores.technical_skills', 'overall_scores.experience',
       'overall_scores.problem_solving', 'overall_scores.communication',
       'overall_scores.cultural_fit', 'overall_scores.overall_suitability',
       'overall_scores.explanation', 'role', 'jd_text', 'extracted_skills',
       'Questionnaire_ID', 'Transcript_ID', 'Role', 'Overall_Suitability',
       'Interviewer_Comments', 'Ratings.Technical_Proficiency',
       'Ratings.Problem_Solving_Ability', 'Ratings.Communication_Skills',
       'Ratings.Cultural_Team_Fit', 'Ratings.Adaptability_Learning',
       'segment_count', 'transcript_embedding', 'skills_vector',
       'rec_Consider', 'rec_Hire', 'rec_Reject'],
      dtype='object')


## 6. Combine Structured Features into a Single Vector
We will combine the following:
- Normalized questionnaire ratings  
- One-hot encoded Recommendation features  
- JD skills binary vector

This creates a final structured feature vector for each candidate.

In [9]:
def combine_features(row):
    # Convert normalized rating columns to a numpy array
    ratings = row[rating_cols].values.astype(float)
    
    # Get one-hot encoded recommendation columns (columns that start with "rec_")
    rec_cols = [col for col in row.index if col.startswith("rec_")]
    rec_features = row[rec_cols].values.astype(float) if rec_cols else np.array([])
    
    # Convert the skills_vector (list) to a numpy array
    skills_vector = np.array(row['skills_vector'])
    
    # Concatenate all parts to form the final structured feature vector
    combined = np.concatenate([ratings, rec_features, skills_vector])
    return combined

df['structured_features'] = df.apply(combine_features, axis=1)

In [10]:
# Check an example structured feature vector and its shape
example_vector = df['structured_features'].iloc[0]
print("Example structured feature vector (first row):")
print(example_vector)
print("Shape of the structured feature vector:", len(example_vector))


Example structured feature vector (first row):
[1. 1. 1. 1. 1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.
 1. 0. 0. 1. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Shape of the structured feature vector: 42


## 7. Save the Dataset with Structured Features

In [12]:
output_file = '../../data/test-final/master_dataset_structured_features.csv'
df.to_csv(output_file, index=False)
print(f"Master dataset with structured features saved to '{output_file}'")

Master dataset with structured features saved to '../../data/test-final/master_dataset_structured_features.csv'
